# 80 — Zones (run_train → mark zones → zoned dataset + ResultsFile)

Thin wrapper around `azlib.validate.run_train` (Tasks 3-7's full pipeline:
fit_stats → per-group regression fit+fuse → Y-sweep/select → R/R levels →
ResultsFile + sidecars, all internal to `run_train` — see that function's
own docstring) for ONE `(tf, direction)`.

`run_train` itself only returns the `ResultsFile` PATH — it does not persist
the actual per-row zoned dataset (it only builds one internally as an
end-to-end pipeline sanity check, then discards it — see `run_train`'s own
source comment). This notebook produces + persists that zoned dataset by
calling `azlib.validate.run_oos` with the JUST-WRITTEN `results_path`
against the SAME (train) dataset — `run_oos` applies the frozen artifacts
with NO refit (see that function's own "Train-only freeze / no leakage"
docstring), which is exactly "mark zones with the just-fit model" applied
to its own training data. (`az_reach_drift_max` on this particular zoned
frame is therefore train-vs-train, near 0 by construction — the real
train→OOS drift check is `90_validate.ipynb`'s job.)

Dataset comes from the ambient environment (`ROOT_FOLDER`/`DATA_ROOT`/
`DATA_SET_NAME`/`PAIR` — see `nb/README.md`); `run_train` needs an
env-FILE PATH argument, so this notebook snapshots the currently-active env
into a temp file for it (`nbutil.snapshot_env_file`, a no-op
re-application of the same values already active).

**Human validation checkpoint (design spec §10):** review this notebook's
charts/report (and `40_regression_classification.ipynb`'s) before treating
the written `ResultsFile` as locked for `90_validate.ipynb`.

In [ ]:
tf = 15
direction = "long"

In [ ]:
import sys
from pathlib import Path

# Bootstrap: notebooks/action_zones/{azlib,nb} onto sys.path. Assumes cwd is
# the repo root (/code) -- true for the experiment Docker service
# (docker-compose.experiment.yml's working_dir: /code; papermill's own
# cwd=None default inherits the invoking process's cwd) and for the
# automated smoke test (tests/test_layer9_notebooks.py sets the papermill
# subprocess's cwd to the repo root to match). helpers.py/indicators.*
# resolve for free via the image's own PYTHONPATH=/code.
_AZ_ROOT = Path.cwd() / "notebooks" / "action_zones"
for _p in (_AZ_ROOT, _AZ_ROOT / "nb"):
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))

import nbutil

In [ ]:
import tempfile

from azlib.validate import run_train, run_oos
from azlib.zones import ResultsFile

In [ ]:
train_env = nbutil.snapshot_env_file(tempfile.mkstemp(suffix=".env")[1])

results_path = run_train(train_env, tf, direction)
zoned_df = run_oos(results_path, train_env)  # apply the just-fit frozen model back onto TRAIN -- see markdown above

rf = ResultsFile.load(results_path)
print(f"results_path={results_path}")
print(f"y={rf.y} tgt_x={rf.tgt_x} sl_x={rf.sl_x} fee={rf.fee}")

In [ ]:
out_dir = nbutil.results_dir(tf, direction, "results")
zoned_dataset_path = f"{out_dir}/zoned_dataset_train.pkl"
zoned_df.to_pickle(zoned_dataset_path)

zone_col = f"az_zone_{direction}_{tf}"
n_zoned = int(zoned_df[zone_col].sum())
summary = {
    "tf": tf,
    "direction": direction,
    "results_path": results_path,
    "zoned_dataset_path": zoned_dataset_path,
    "n_rows": len(zoned_df),
    "n_zoned": n_zoned,
    "frac_zoned": (n_zoned / len(zoned_df)) if len(zoned_df) else None,
    "y": rf.y,
    "tgt_x": rf.tgt_x,
    "sl_x": rf.sl_x,
}
summary_path = nbutil.write_json_report(f"{out_dir}/zones_summary.json", summary)
print(summary)